# Transformer Architecture from Scratch
A simple and intuitive guide to understanding, building, and training a small Transformer model using PyTorch.

### Core Architecture & Math
Introduced in *Attention Is All You Need* (Vaswani et al., 2017), the Transformer replaces recurrent layers with **Self-Attention** mechanisms.

#### 1. Scaled Dot-Product Attention
Given queries $Q$, keys $K$, and values $V$:
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q K^T}{\sqrt{d_k}}\right) V$$
- **Scaling Factor ($\sqrt{d_k}$)**: Prevents dot products from growing excessively large, avoiding vanishing gradients in softmax.
- **Causal Mask**: Ensures position $i$ cannot attend to future positions $j > i$ during autoregressive language modeling.

#### 2. Multi-Head Attention (MHA)
Splits model dimension $d_{\text{model}}$ into $h$ heads ($d_k = d_{\text{model}} / h$):
$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \dots, \text{head}_h) W^O$$
$$\text{head}_i = \text{Attention}(Q W_i^Q, K W_i^K, V W_i^V)$$

#### 3. Positional Encoding
Since self-attention is permutation-equivariant, positional information is added to input embeddings using sinusoidal functions:
$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)$$
$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)$$

#### 4. Feed-Forward Network & Residual Connections
- **Position-wise FFN**: $FFN(x) = \max(0, x W_1 + b_1) W_2 + b_2$
- **Residual Connection & LayerNorm**: $\text{LayerNorm}(x + \text{SubLayer}(x))$

## Part 1: Scaled Dot-Product Attention & Multi-Head Attention
Step-by-step implementation of `ScaledDotProductAttention` and `MultiHeadAttention` modules in PyTorch.

In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# Set seed for reproducibility
torch.manual_seed(42)

class ScaledDotProductAttention(nn.Module):
    """
    Computes Scaled Dot-Product Attention:
    Attention(Q, K, V) = softmax(Q K^T / sqrt(d_k)) V
    """
    def __init__(self, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

    def forward(self, q, k, v, mask=None):
        # q, k, v shape: (batch_size, n_heads, seq_len, d_k)
        d_k = q.size(-1)
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_k)
        
        if mask is not None:
            # Mask out forbidden positions
            scores = scores.masked_fill(mask == 0, -1e9)
            
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        output = torch.matmul(attn_weights, v)
        return output, attn_weights

class MultiHeadAttention(nn.Module):
    """
    Multi-Head Attention module that projects Q, K, V into multiple heads,
    computes attention in parallel, and projects back.
    """
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        
        self.attention = ScaledDotProductAttention(dropout=dropout)
        
    def forward(self, q, k, v, mask=None):
        batch_size, seq_len, _ = q.size()
        
        # 1. Linear projection & reshape to (batch_size, n_heads, seq_len, d_k)
        q_heads = self.q_proj(q).view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        k_heads = self.k_proj(k).view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        v_heads = self.v_proj(v).view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        
        # 2. Scaled Dot-Product Attention across all heads
        attn_out, attn_weights = self.attention(q_heads, k_heads, v_heads, mask=mask)
        
        # 3. Concatenate heads back to (batch_size, seq_len, d_model)
        attn_out = attn_out.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        
        # 4. Final linear projection
        output = self.out_proj(attn_out)
        return output, attn_weights

# Quick sanity check
batch_size, seq_len, d_model, n_heads = 2, 5, 16, 4
mha = MultiHeadAttention(d_model=d_model, n_heads=n_heads)
dummy_x = torch.randn(batch_size, seq_len, d_model)

out, weights = mha(dummy_x, dummy_x, dummy_x)
print("Input shape:", list(dummy_x.shape))
print("MHA Output shape:", list(out.shape))
print("Attention weights shape:", list(weights.shape))


Input shape: [2, 5, 16]
MHA Output shape: [2, 5, 16]
Attention weights shape: [2, 4, 5, 5]


## Part 2: Positional Encoding & Transformer Block
Adding sinusoidal positional encodings and assembling a complete `TransformerBlock`.

In [2]:
class PositionalEncoding(nn.Module):
    """
    Injects sinusoidal positional encodings into token embeddings.
    """
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        self.register_buffer('pe', pe.unsqueeze(0))
        
    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

class PositionwiseFeedForward(nn.Module):
    """
    Position-wise Feed-Forward Network: FFN(x) = max(0, x W1 + b1) W2 + b2
    """
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        return self.linear2(self.dropout(F.relu(self.linear1(x))))

class TransformerBlock(nn.Module):
    """
    A standard Transformer Block combining MHA, FFN, LayerNorm, and Residuals.
    """
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.mha = MultiHeadAttention(d_model, n_heads, dropout)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, mask=None):
        # Pre-LN architecture
        attn_out, weights = self.mha(self.norm1(x), self.norm1(x), self.norm1(x), mask=mask)
        x = x + self.dropout(attn_out)
        
        ffn_out = self.ffn(self.norm2(x))
        x = x + self.dropout(ffn_out)
        return x, weights

# Quick sanity check
pe = PositionalEncoding(d_model=16)
block = TransformerBlock(d_model=16, n_heads=4, d_ff=64)

x_pos = pe(dummy_x)
block_out, _ = block(x_pos)
print("Positional Encoding Output shape:", list(x_pos.shape))
print("Transformer Block Output shape:", list(block_out.shape))


Positional Encoding Output shape: [2, 5, 16]
Transformer Block Output shape: [2, 5, 16]


## Part 3: Assembling a Small Decoder Transformer Language Model
We create `SmallTransformerLM` using stacked Transformer blocks and a causal lower-triangular mask.

In [3]:
class SmallTransformerLM(nn.Module):
    """
    Small Decoder-only Transformer Language Model for Causal Sequence Modeling.
    """
    def __init__(self, vocab_size, d_model=32, n_heads=4, num_layers=2, d_ff=128, max_seq_len=64, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len=max_seq_len)
        
        self.layers = nn.ModuleList([
            TransformerBlock(d_model=d_model, n_heads=n_heads, d_ff=d_ff, dropout=dropout)
            for _ in range(num_layers)
        ])
        
        self.final_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)
        
    def generate_causal_mask(self, seq_len, device):
        mask = torch.tril(torch.ones((seq_len, seq_len), device=device)).unsqueeze(0).unsqueeze(0)
        return mask
        
    def forward(self, input_ids):
        batch_size, seq_len = input_ids.size()
        device = input_ids.device
        
        x = self.token_embedding(input_ids) * math.sqrt(self.d_model)
        x = self.pos_encoding(x)
        
        mask = self.generate_causal_mask(seq_len, device)
        
        for layer in self.layers:
            x, _ = layer(x, mask=mask)
            
        x = self.final_norm(x)
        logits = self.lm_head(x)
        return logits

# Instantiate small model
vocab_size = 15
model = SmallTransformerLM(vocab_size=vocab_size, d_model=32, n_heads=4, num_layers=2, d_ff=128)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total Model Parameters: {total_params:,}")


Total Model Parameters: 26,447


## Part 4: Training the Transformer on a Synthetic Pattern Task
We train our model to predict the next number in a repeating sequence pattern: `[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 1, 2, ...]`.

In [4]:
# Create synthetic sequence dataset
sequence_pattern = list(range(1, 11))  # [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
full_sequence = sequence_pattern * 50    # Length 500 sequence
seq_len = 16

X_data, Y_data = [], []
for i in range(len(full_sequence) - seq_len):
    X_data.append(full_sequence[i : i + seq_len])
    Y_data.append(full_sequence[i + 1 : i + seq_len + 1])

X_tensor = torch.tensor(X_data, dtype=torch.long)
Y_tensor = torch.tensor(Y_data, dtype=torch.long)

print(f"Dataset shape: X={list(X_tensor.shape)}, Y={list(Y_tensor.shape)}")

# Re-instantiate model with dropout=0 for fast deterministic training
torch.manual_seed(42)
model = SmallTransformerLM(vocab_size=15, d_model=32, n_heads=4, num_layers=2, d_ff=128, dropout=0.0)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.003)
criterion = nn.CrossEntropyLoss()

# Training loop
epochs = 80
print("\n--- Starting Training ---")
for epoch in range(1, epochs + 1):
    model.train()
    optimizer.zero_grad()
    
    logits = model(X_tensor)  # (batch_size, seq_len, vocab_size)
    loss = criterion(logits.view(-1, 15), Y_tensor.view(-1))
    
    loss.backward()
    optimizer.step()
    
    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:2d}/{epochs} | Loss: {loss.item():.4f}")

print("Training completed successfully!")


Dataset shape: X=[484, 16], Y=[484, 16]

--- Starting Training ---
Epoch  1/80 | Loss: 2.7766
Epoch 10/80 | Loss: 1.4146
Epoch 20/80 | Loss: 0.4400
Epoch 30/80 | Loss: 0.1480
Epoch 40/80 | Loss: 0.0732
Epoch 50/80 | Loss: 0.0444
Epoch 60/80 | Loss: 0.0310
Epoch 70/80 | Loss: 0.0236
Epoch 80/80 | Loss: 0.0190
Training completed successfully!


## Part 5: Autoregressive Sequence Generation
Testing autoregressive token generation using prompt `[1, 2, 3]`.

In [5]:
def generate(model, prompt, max_new_tokens=12):
    model.eval()
    generated = prompt.copy()
    
    with torch.no_grad():
        for _ in range(max_new_tokens):
            input_ids = torch.tensor([generated], dtype=torch.long)
            logits = model(input_ids)
            next_token_logits = logits[0, -1, :]
            next_token = torch.argmax(next_token_logits).item()
            generated.append(next_token)
            
    return generated

prompt = [1, 2, 3]
output_seq = generate(model, prompt, max_new_tokens=12)
print("Prompt Sequence:   ", prompt)
print("Generated Sequence:", output_seq)


Prompt Sequence:    [1, 2, 3]
Generated Sequence: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 1, 2, 3, 4, 5]
